In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ==================================================================================================
# 10_route_A_manuscript_figures.ipynb
# COMPLETE SINGLE-CELL, POST-HOC FIGURE GENERATION
# ==================================================================================================
#
# PURPOSE
# -------
# Generate publication-ready manuscript figures from locked Route A outputs only.
#
# THIS NOTEBOOK DOES NOT:
#   - train any model
#   - load any model checkpoint
#   - access raw LIMUC images
#   - perform model inference
#   - change the locked model
#   - select a model, seed, fold, threshold, aggregation rule, or reference rule
#   - read any pre-Route-A legacy result
#
# MAIN FIGURES
# ------------
# Figure 1:
#   Route A development-only reconstruction and locked official-test workflow
#
# Figure 2:
#   Locked Swin official-test patient- and image-level confusion matrices
#
# Figure 3:
#   Patient-level per-class precision, recall, and F1 with 95% bootstrap CIs
#
# Figure 4:
#   Prediction-aggregation and derived-reference sensitivity
#
# SUPPLEMENTARY FIGURES
# ---------------------
# Supplementary Figure S1:
#   Raw-count confusion matrices
#
# Supplementary Figure S2:
#   Image- versus patient-level per-class point estimates
#
# Supplementary Figure S3:
#   Uniform three-seed Swin-versus-ViT development benchmark
#
# Supplementary Figure S4:
#   Four-condition Route A factorial ablation, when a parseable RA07 summary exists
#
# OUTPUT ROOT
# -----------
# ${EGLIMUC_WORKSPACE}/LIMUC/ViT2/revision_2026/
#     10_route_A_manuscript_figures/
#
# ==================================================================================================


# ==================================================================================================
# 0. Install missing lightweight dependencies
# ==================================================================================================

import sys
import subprocess
import importlib.util


REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}


for import_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )


# ==================================================================================================
# 1. Imports
# ==================================================================================================

import os
import re
import json
import math
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import seaborn as sns

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from IPython.display import display


warnings.filterwarnings("ignore")


# ==================================================================================================
# 2. Mount Google Drive
# ==================================================================================================

try:
    from google.colab import drive

    (drive.mount(
        "/content/drive",
        force_remount=False,
    ) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)

except Exception as mount_error:
    print("Google Drive was not mounted automatically.")
    print("Reason:", repr(mount_error))


# ==================================================================================================
# 3. Constants
# ==================================================================================================

NUM_CLASSES = 4

CLASS_IDS = [
    0,
    1,
    2,
    3,
]

CLASS_NAMES = {
    0: "Mayo 0",
    1: "Mayo 1",
    2: "Mayo 2",
    3: "Mayo 3",
}

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


EXPECTED_TEST_IMAGES = 1686
EXPECTED_TEST_PATIENTS = 86

EXPECTED_PATIENT_SUPPORT = {
    0: 56,
    1: 19,
    2: 5,
    3: 6,
}

EXPECTED_SWIN_TEST_PATIENT_QWK = 0.7847
EXPECTED_SWIN_TEST_IMAGE_QWK = 0.8689

PATIENT_QWK_TOLERANCE = 0.020
IMAGE_QWK_TOLERANCE = 0.035

EXPECTED_LOCK_FINGERPRINT = (
    "8852c67de6e530e8e6e15b51e94d68be8f3ddffb6d364b328791cf5269e7ef91"
)

BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20260910

TRIM_FRACTION = 0.10
MIN_IMAGES_FOR_CONFIDENCE_TRIM = 5

PNG_DPI = 600

SAVE_PNG = True
SAVE_SVG = True
SAVE_PDF = True

SHOW_FIGURES = True


# Colorblind-friendly palette
PRIMARY_COLOR = "#0072B2"
COMPARATOR_COLOR = "#D55E00"
SECONDARY_COLOR = "#56B4E9"
ACCENT_COLOR = "#E69F00"
GREEN_COLOR = "#009E73"
PURPLE_COLOR = "#CC79A7"
RED_COLOR = "#C44E52"
NEUTRAL_COLOR = "#7A7A7A"
LIGHT_NEUTRAL = "#D9D9D9"
DARK_COLOR = "#222222"


mpl.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "figure.titlesize": 15,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

sns.set_theme(
    style="whitegrid",
    context="paper",
)


# ==================================================================================================
# 4. Resolve Route A directories
# ==================================================================================================

REVISION_ROOT_CANDIDATES = [
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
    ),
    Path(
        (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
    ),
]


REVISION_ROOT = next(
    (
        path
        for path in REVISION_ROOT_CANDIDATES
        if path.exists()
    ),
    None,
)


if REVISION_ROOT is None:
    checked_paths = "\n".join(
        str(path)
        for path in REVISION_ROOT_CANDIDATES
    )

    raise FileNotFoundError(
        "Could not locate the Route A revision directory.\n"
        "Checked:\n"
        + checked_paths
    )


def resolve_stage_directory(
    prefix,
    preferred_names,
    required=True,
):
    for preferred_name in preferred_names:
        preferred_path = (
            REVISION_ROOT
            / preferred_name
        )

        if preferred_path.exists():
            return preferred_path

    candidates = sorted(
        [
            path
            for path in REVISION_ROOT.iterdir()
            if (
                path.is_dir()
                and
                path.name.startswith(prefix)
            )
        ],
        key=lambda path: (
            path.stat().st_mtime,
            path.name,
        ),
    )

    if candidates:
        return candidates[-1]

    if required:
        raise FileNotFoundError(
            "Could not locate a Route A directory beginning with "
            + repr(prefix)
            + " under:\n"
            + str(REVISION_ROOT)
        )

    return None


RA01 = resolve_stage_directory(
    prefix="01_",
    preferred_names=[
        "01_vit_loss_screening_development_only",
    ],
)

RA03 = resolve_stage_directory(
    prefix="03_",
    preferred_names=[
        "03_patient_aggregation_and_reference_sensitivity",
    ],
)

RA06 = resolve_stage_directory(
    prefix="06_",
    preferred_names=[
        "06_backbone_benchmark_development_only_v1",
        "06_backbone_benchmark_development_only",
    ],
)

RA07 = resolve_stage_directory(
    prefix="07_",
    preferred_names=[
        "07_locked_swin_ablation_development_only_v1",
        "07_locked_swin_ablation_development_only",
        "07_swin_factorial_ablation_development_only",
    ],
    required=False,
)

RA08 = resolve_stage_directory(
    prefix="08_",
    preferred_names=[
        "08_final_model_lock_before_test",
    ],
)

RA09 = resolve_stage_directory(
    prefix="09_",
    preferred_names=[
        "09_locked_official_test_evaluation",
    ],
)


OUTPUT_ROOT = (
    REVISION_ROOT
    / "10_route_A_manuscript_figures"
)

FIGURE_ROOT = (
    OUTPUT_ROOT
    / "figures"
)

PNG_DIR = (
    FIGURE_ROOT
    / "png"
)

SVG_DIR = (
    FIGURE_ROOT
    / "svg"
)

PDF_DIR = (
    FIGURE_ROOT
    / "pdf"
)

SOURCE_DATA_DIR = (
    OUTPUT_ROOT
    / "source_data"
)

MANIFEST_DIR = (
    OUTPUT_ROOT
    / "manifests"
)


for directory in [
    OUTPUT_ROOT,
    FIGURE_ROOT,
    PNG_DIR,
    SVG_DIR,
    PDF_DIR,
    SOURCE_DATA_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("=" * 125)
print("10_route_A_manuscript_figures")
print("=" * 125)

print("Revision root:", REVISION_ROOT)
print("RA01:", RA01)
print("RA03:", RA03)
print("RA06:", RA06)
print("RA07:", RA07)
print("RA08:", RA08)
print("RA09:", RA09)
print("Output root:", OUTPUT_ROOT)


# ==================================================================================================
# 5. Safety checks
# ==================================================================================================

LEGACY_PATH_FRAGMENT = "/LIMUC/ViT/"


for route_path in [
    RA01,
    RA03,
    RA06,
    RA08,
    RA09,
]:
    normalized_path = str(route_path).replace(
        "\\",
        "/",
    )

    if LEGACY_PATH_FRAGMENT in normalized_path:
        raise RuntimeError(
            "A legacy pre-Route-A directory was selected:\n"
            + normalized_path
        )


TRAINING_PERFORMED = False
CHECKPOINTS_LOADED = False
RAW_IMAGES_ACCESSED = False
MODEL_INFERENCE_PERFORMED = False
MODEL_SELECTION_PERFORMED = False
LEGACY_RESULTS_USED = False


print("")
print("Safety status:")
print("  Training performed:", TRAINING_PERFORMED)
print("  Checkpoints loaded:", CHECKPOINTS_LOADED)
print("  Raw images accessed:", RAW_IMAGES_ACCESSED)
print("  Model inference performed:", MODEL_INFERENCE_PERFORMED)
print("  Model selection performed:", MODEL_SELECTION_PERFORMED)
print("  Legacy results used:", LEGACY_RESULTS_USED)


# ==================================================================================================
# 6. File-output helpers
# ==================================================================================================

def atomic_csv_save(
    dataframe,
    path,
    index=False,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    dataframe.to_csv(
        temporary_path,
        index=index,
    )

    os.replace(
        temporary_path,
        path,
    )


def atomic_json_save(
    payload,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            default=str,
        )

        file.flush()
        os.fsync(file.fileno())

    os.replace(
        temporary_path,
        path,
    )


def atomic_text_save(
    text,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".temporary"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(text)
        file.flush()
        os.fsync(file.fileno())

    os.replace(
        temporary_path,
        path,
    )


def sha256_file(
    path,
    block_size=8 * 1024 * 1024,
):
    path = Path(path)

    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(block_size)

            if not block:
                break

            hasher.update(block)

    return hasher.hexdigest()


for abandoned_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass


# ==================================================================================================
# 7. Figure-output helpers
# ==================================================================================================

USED_SOURCE_FILES = {}
GENERATED_FIGURES = []


def register_source(
    label,
    path,
):
    if path is None:
        return

    path = Path(path)

    if path.exists():
        USED_SOURCE_FILES[label] = path


def save_figure(
    figure,
    stem,
    description,
):
    output_paths = []

    if SAVE_PNG:
        png_path = (
            PNG_DIR
            / f"{stem}.png"
        )

        figure.savefig(
            png_path,
            dpi=PNG_DPI,
            bbox_inches="tight",
            facecolor="white",
        )

        output_paths.append(png_path)

    if SAVE_SVG:
        svg_path = (
            SVG_DIR
            / f"{stem}.svg"
        )

        figure.savefig(
            svg_path,
            bbox_inches="tight",
            facecolor="white",
        )

        output_paths.append(svg_path)

    if SAVE_PDF:
        pdf_path = (
            PDF_DIR
            / f"{stem}.pdf"
        )

        figure.savefig(
            pdf_path,
            bbox_inches="tight",
            facecolor="white",
        )

        output_paths.append(pdf_path)

    for output_path in output_paths:
        GENERATED_FIGURES.append(
            {
                "figure_stem": stem,
                "description": description,
                "file_type": output_path.suffix.lower(),
                "path": str(output_path),
            }
        )

    if SHOW_FIGURES:
        plt.show()

    plt.close(figure)


# ==================================================================================================
# 8. General dataframe helpers
# ==================================================================================================

def normalize_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(value).strip().lower(),
    ).strip("_")


def find_column(
    dataframe,
    aliases,
):
    normalized_to_original = {
        normalize_name(column): column
        for column in dataframe.columns
    }

    for alias in aliases:
        normalized_alias = normalize_name(alias)

        if normalized_alias in normalized_to_original:
            return normalized_to_original[
                normalized_alias
            ]

    return None


def identify_probability_columns(
    dataframe,
):
    probability_map = {}

    for column in dataframe.columns:
        normalized = normalize_name(column)

        patterns = [
            r"^prob_?([0-3])$",
            r"^probability_?([0-3])$",
            r"^p_?([0-3])$",
            r"^class_?([0-3])_?prob$",
            r"^class_?([0-3])_?probability$",
            r"^mayo_?([0-3])_?prob$",
            r"^mayo_?([0-3])_?probability$",
            r".*_prob_?([0-3])$",
            r".*_probability_?([0-3])$",
        ]

        for pattern in patterns:
            match = re.match(
                pattern,
                normalized,
            )

            if match is not None:
                class_id = int(
                    match.group(1)
                )

                probability_map[
                    class_id
                ] = column

                break

    if set(probability_map.keys()) == set(CLASS_IDS):
        return [
            probability_map[class_id]
            for class_id in CLASS_IDS
        ]

    return []


def filter_to_swin_rows(
    dataframe,
):
    dataframe = dataframe.copy()

    possible_system_columns = [
        "backbone",
        "backbone_name",
        "model",
        "model_name",
        "candidate",
        "candidate_name",
        "system",
        "system_name",
        "configuration",
        "representative_configuration",
        "role",
    ]

    for possible_name in possible_system_columns:
        column = find_column(
            dataframe,
            aliases=[
                possible_name,
            ],
        )

        if column is None:
            continue

        swin_mask = dataframe[
            column
        ].astype(str).str.contains(
            "swin",
            case=False,
            na=False,
        )

        if swin_mask.any():
            dataframe = dataframe.loc[
                swin_mask
            ].copy()

    return dataframe.reset_index(
        drop=True
    )


def safe_read_csv(
    path,
):
    return pd.read_csv(
        path,
        low_memory=False,
    )


# ==================================================================================================
# 9. Validate the pre-test lock receipt
# ==================================================================================================

LOCK_SEARCH_FILES = []

for extension in [
    "*.json",
    "*.txt",
]:
    LOCK_SEARCH_FILES.extend(
        sorted(
            RA08.rglob(extension)
        )
    )


LOCK_FINGERPRINT_VALIDATED = False
LOCK_FINGERPRINT_SOURCE = None


for candidate_path in LOCK_SEARCH_FILES:
    try:
        text = candidate_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

    except Exception:
        continue

    if EXPECTED_LOCK_FINGERPRINT in text:
        LOCK_FINGERPRINT_VALIDATED = True
        LOCK_FINGERPRINT_SOURCE = candidate_path
        register_source(
            "RA08 pre-test lock fingerprint",
            candidate_path,
        )
        break


print("")
print("Pre-test lock fingerprint expected:")
print(EXPECTED_LOCK_FINGERPRINT)

print("Fingerprint found in RA08 artifacts:")
print(LOCK_FINGERPRINT_VALIDATED)

print("Fingerprint source:")
print(LOCK_FINGERPRINT_SOURCE)


if not LOCK_FINGERPRINT_VALIDATED:
    print(
        "WARNING: The expected fingerprint was not found as text. "
        "The figure notebook will continue, but the RA08 lock directory "
        "should be checked before public release."
    )


# ==================================================================================================
# 10. Metric and patient-reference helpers
# ==================================================================================================

def mode_low(
    values,
):
    values = np.asarray(
        values,
        dtype=int,
    )

    counts = np.bincount(
        values,
        minlength=NUM_CLASSES,
    )

    return int(
        np.flatnonzero(
            counts == counts.max()
        )[0]
    )


def mode_high(
    values,
):
    values = np.asarray(
        values,
        dtype=int,
    )

    counts = np.bincount(
        values,
        minlength=NUM_CLASSES,
    )

    return int(
        np.flatnonzero(
            counts == counts.max()
        )[-1]
    )


def median_low(
    values,
):
    values = np.sort(
        np.asarray(
            values,
            dtype=int,
        )
    )

    index = (
        len(values) - 1
    ) // 2

    return int(
        values[index]
    )


def median_high(
    values,
):
    values = np.sort(
        np.asarray(
            values,
            dtype=int,
        )
    )

    index = len(values) // 2

    return int(
        values[index]
    )


def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    return {
        "qwk": float(
            cohen_kappa_score(
                y_true,
                y_pred,
                labels=CLASS_IDS,
                weights="quadratic",
            )
        ),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=CLASS_IDS,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def derive_patient_predictions(
    image_dataframe,
):
    rows = []

    for patient_id, group in image_dataframe.groupby(
        "patient_id",
        sort=True,
    ):
        mean_probabilities = group[
            PROBABILITY_COLUMNS
        ].mean(
            axis=0
        ).to_numpy(
            dtype=float
        )

        mean_probabilities = (
            mean_probabilities
            / mean_probabilities.sum()
        )

        image_labels = group[
            "y_true"
        ].to_numpy(
            dtype=int
        )

        row = {
            "patient_id": int(patient_id),
            "y_true": mode_low(image_labels),
            "y_pred": int(
                np.argmax(
                    mean_probabilities
                )
            ),
            "n_images": int(
                len(group)
            ),
        }

        for class_id in CLASS_IDS:
            row[
                f"prob_{class_id}"
            ] = float(
                mean_probabilities[class_id]
            )

        rows.append(row)

    return pd.DataFrame(
        rows
    ).sort_values(
        "patient_id"
    ).reset_index(
        drop=True
    )


# ==================================================================================================
# 11. Standardize a candidate RA09 image-prediction CSV
# ==================================================================================================

IMAGE_TRUE_ALIASES = [
    "y_true",
    "true_label",
    "image_label",
    "label",
    "target",
    "mes_label",
    "mayo_label",
    "mayo_class",
]

PATIENT_ID_ALIASES = [
    "patient_id",
    "patient",
    "pid",
    "group_id",
]

PATH_ALIASES = [
    "path",
    "image_path",
    "filepath",
    "file_path",
    "filename",
]


def standardize_image_prediction_candidate(
    path,
):
    dataframe = safe_read_csv(path)
    dataframe = filter_to_swin_rows(dataframe)

    true_column = find_column(
        dataframe,
        aliases=IMAGE_TRUE_ALIASES,
    )

    patient_column = find_column(
        dataframe,
        aliases=PATIENT_ID_ALIASES,
    )

    path_column = find_column(
        dataframe,
        aliases=PATH_ALIASES,
    )

    probability_source_columns = (
        identify_probability_columns(
            dataframe
        )
    )

    if true_column is None:
        return None

    if patient_column is None:
        return None

    if len(probability_source_columns) != NUM_CLASSES:
        return None

    output = pd.DataFrame()

    output[
        "patient_id"
    ] = pd.to_numeric(
        dataframe[patient_column],
        errors="coerce",
    )

    output[
        "y_true"
    ] = pd.to_numeric(
        dataframe[true_column],
        errors="coerce",
    )

    if path_column is not None:
        output[
            "path"
        ] = dataframe[
            path_column
        ].astype(str)

    else:
        output[
            "path"
        ] = [
            f"row_{row_index:06d}"
            for row_index in range(
                len(dataframe)
            )
        ]

    for class_id, source_column in enumerate(
        probability_source_columns
    ):
        output[
            f"prob_{class_id}"
        ] = pd.to_numeric(
            dataframe[source_column],
            errors="coerce",
        )

    output = output.dropna(
        subset=[
            "patient_id",
            "y_true",
            "prob_0",
            "prob_1",
            "prob_2",
            "prob_3",
        ]
    ).copy()

    output[
        "patient_id"
    ] = output[
        "patient_id"
    ].astype(int)

    output[
        "y_true"
    ] = output[
        "y_true"
    ].astype(int)

    probability_matrix = output[
        PROBABILITY_COLUMNS
    ].to_numpy(
        dtype=float
    )

    probability_matrix = np.clip(
        probability_matrix,
        0.0,
        None,
    )

    row_sums = probability_matrix.sum(
        axis=1,
        keepdims=True,
    )

    if np.any(row_sums <= 0):
        return None

    probability_matrix = (
        probability_matrix
        / row_sums
    )

    output[
        PROBABILITY_COLUMNS
    ] = probability_matrix

    output[
        "y_pred"
    ] = np.argmax(
        probability_matrix,
        axis=1,
    ).astype(int)

    return output.reset_index(
        drop=True
    )


# ==================================================================================================
# 12. Discover the locked Swin official-test image prediction file
# ==================================================================================================

candidate_records = []
accepted_candidates = []


for csv_path in sorted(
    RA09.rglob("*.csv")
):
    try:
        standardized = (
            standardize_image_prediction_candidate(
                csv_path
            )
        )

    except Exception as error:
        candidate_records.append(
            {
                "path": str(csv_path),
                "status": "read_error",
                "n_rows": np.nan,
                "n_patients": np.nan,
                "image_qwk": np.nan,
                "patient_qwk": np.nan,
                "score": -9999,
                "error": repr(error),
            }
        )

        continue

    if standardized is None:
        candidate_records.append(
            {
                "path": str(csv_path),
                "status": "not_prediction_format",
                "n_rows": np.nan,
                "n_patients": np.nan,
                "image_qwk": np.nan,
                "patient_qwk": np.nan,
                "score": -9999,
                "error": "",
            }
        )

        continue

    n_rows = int(
        len(standardized)
    )

    n_patients = int(
        standardized[
            "patient_id"
        ].nunique()
    )

    status = "candidate"

    image_qwk = np.nan
    patient_qwk = np.nan
    score = 0.0

    if (
        n_rows == EXPECTED_TEST_IMAGES
        and
        n_patients == EXPECTED_TEST_PATIENTS
    ):
        patient_candidate = derive_patient_predictions(
            standardized
        )

        image_candidate_metrics = compute_metrics(
            standardized[
                "y_true"
            ],
            standardized[
                "y_pred"
            ],
        )

        patient_candidate_metrics = compute_metrics(
            patient_candidate[
                "y_true"
            ],
            patient_candidate[
                "y_pred"
            ],
        )

        image_qwk = image_candidate_metrics["qwk"]
        patient_qwk = patient_candidate_metrics["qwk"]

        patient_support = (
            patient_candidate[
                "y_true"
            ]
            .value_counts()
            .sort_index()
            .reindex(
                CLASS_IDS,
                fill_value=0,
            )
            .to_dict()
        )

        name_lower = csv_path.name.lower()

        if "swin" in name_lower:
            score += 40

        if "official" in name_lower:
            score += 20

        if "test" in name_lower:
            score += 20

        if "image" in name_lower:
            score += 15

        if "prediction" in name_lower:
            score += 15

        if "ensemble" in name_lower:
            score += 10

        if "locked" in name_lower:
            score += 10

        if "vit" in name_lower and "swin" not in name_lower:
            score -= 50

        image_difference = abs(
            image_qwk
            - EXPECTED_SWIN_TEST_IMAGE_QWK
        )

        patient_difference = abs(
            patient_qwk
            - EXPECTED_SWIN_TEST_PATIENT_QWK
        )

        score -= 1000 * patient_difference
        score -= 200 * image_difference

        valid_metrics = (
            patient_difference <= PATIENT_QWK_TOLERANCE
            and
            image_difference <= IMAGE_QWK_TOLERANCE
        )

        valid_support = (
            patient_support == EXPECTED_PATIENT_SUPPORT
        )

        if valid_metrics and valid_support:
            status = "accepted_locked_swin_candidate"

            accepted_candidates.append(
                {
                    "path": csv_path,
                    "dataframe": standardized,
                    "score": score,
                    "image_qwk": image_qwk,
                    "patient_qwk": patient_qwk,
                }
            )

        else:
            status = "metrics_or_support_do_not_match_lock"

    candidate_records.append(
        {
            "path": str(csv_path),
            "status": status,
            "n_rows": n_rows,
            "n_patients": n_patients,
            "image_qwk": image_qwk,
            "patient_qwk": patient_qwk,
            "score": score,
            "error": "",
        }
    )


candidate_catalog_df = pd.DataFrame(
    candidate_records
)


atomic_csv_save(
    candidate_catalog_df,
    MANIFEST_DIR
    / "ra09_image_prediction_file_discovery.csv",
    index=False,
)


if not accepted_candidates:
    top_candidates = (
        candidate_catalog_df
        .sort_values(
            "score",
            ascending=False,
        )
        .head(20)
    )

    print("")
    print("Top file-discovery candidates:")
    display(top_candidates)

    raise FileNotFoundError(
        "Could not identify a locked Swin official-test image-prediction CSV "
        "that reproduces the expected Route A counts and QWK values.\n"
        "Inspect:\n"
        + str(
            MANIFEST_DIR
            / "ra09_image_prediction_file_discovery.csv"
        )
    )


accepted_candidates = sorted(
    accepted_candidates,
    key=lambda record: (
        -record["score"],
        str(record["path"]),
    ),
)


selected_candidate = accepted_candidates[0]

IMAGE_PREDICTION_FILE = selected_candidate[
    "path"
]

image_predictions_df = selected_candidate[
    "dataframe"
]


register_source(
    "RA09 locked Swin official-test image predictions",
    IMAGE_PREDICTION_FILE,
)


print("")
print("Selected locked Swin image prediction file:")
print(IMAGE_PREDICTION_FILE)


# ==================================================================================================
# 13. Derive and validate locked patient predictions
# ==================================================================================================

patient_predictions_df = derive_patient_predictions(
    image_predictions_df
)


image_metrics = compute_metrics(
    image_predictions_df[
        "y_true"
    ],
    image_predictions_df[
        "y_pred"
    ],
)

patient_metrics = compute_metrics(
    patient_predictions_df[
        "y_true"
    ],
    patient_predictions_df[
        "y_pred"
    ],
)


observed_patient_support = (
    patient_predictions_df[
        "y_true"
    ]
    .value_counts()
    .sort_index()
    .reindex(
        CLASS_IDS,
        fill_value=0,
    )
    .to_dict()
)


if len(image_predictions_df) != EXPECTED_TEST_IMAGES:
    raise RuntimeError(
        "Expected "
        + str(EXPECTED_TEST_IMAGES)
        + " images, but found "
        + str(len(image_predictions_df))
        + "."
    )


if len(patient_predictions_df) != EXPECTED_TEST_PATIENTS:
    raise RuntimeError(
        "Expected "
        + str(EXPECTED_TEST_PATIENTS)
        + " patients, but found "
        + str(len(patient_predictions_df))
        + "."
    )


if observed_patient_support != EXPECTED_PATIENT_SUPPORT:
    raise RuntimeError(
        "The modal-low patient support does not match the Route A lock.\n"
        "Expected: "
        + repr(EXPECTED_PATIENT_SUPPORT)
        + "\nObserved: "
        + repr(observed_patient_support)
    )


if abs(
    patient_metrics["qwk"]
    - EXPECTED_SWIN_TEST_PATIENT_QWK
) > PATIENT_QWK_TOLERANCE:
    raise RuntimeError(
        "Patient QWK does not reproduce the locked Route A Swin result."
    )


if abs(
    image_metrics["qwk"]
    - EXPECTED_SWIN_TEST_IMAGE_QWK
) > IMAGE_QWK_TOLERANCE:
    raise RuntimeError(
        "Image QWK does not reproduce the locked Route A Swin result."
    )


print("")
print("=" * 125)
print("LOCKED SWIN OFFICIAL-TEST VALIDATION")
print("=" * 125)

print("Images:", len(image_predictions_df))
print("Patients:", len(patient_predictions_df))
print("Patient support:", observed_patient_support)

print("")
print("Image metrics:")
print(
    json.dumps(
        image_metrics,
        indent=2,
    )
)

print("")
print("Patient metrics:")
print(
    json.dumps(
        patient_metrics,
        indent=2,
    )
)


atomic_csv_save(
    image_predictions_df,
    SOURCE_DATA_DIR
    / "locked_swin_official_test_image_predictions_for_figures.csv",
    index=False,
)

atomic_csv_save(
    patient_predictions_df,
    SOURCE_DATA_DIR
    / "locked_swin_official_test_patient_predictions_for_figures.csv",
    index=False,
)


# ==================================================================================================
# 14. Confusion matrices
# ==================================================================================================

patient_confusion_raw = confusion_matrix(
    patient_predictions_df[
        "y_true"
    ],
    patient_predictions_df[
        "y_pred"
    ],
    labels=CLASS_IDS,
)

image_confusion_raw = confusion_matrix(
    image_predictions_df[
        "y_true"
    ],
    image_predictions_df[
        "y_pred"
    ],
    labels=CLASS_IDS,
)


def row_normalize_confusion(
    matrix,
):
    matrix = np.asarray(
        matrix,
        dtype=float,
    )

    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )

    return np.divide(
        matrix,
        row_sums,
        out=np.zeros_like(
            matrix,
            dtype=float,
        ),
        where=row_sums != 0,
    )


patient_confusion_normalized = (
    row_normalize_confusion(
        patient_confusion_raw
    )
)

image_confusion_normalized = (
    row_normalize_confusion(
        image_confusion_raw
    )
)


def confusion_dataframe(
    matrix,
):
    return pd.DataFrame(
        matrix,
        index=[
            f"true_{class_id}"
            for class_id in CLASS_IDS
        ],
        columns=[
            f"pred_{class_id}"
            for class_id in CLASS_IDS
        ],
    )


atomic_csv_save(
    confusion_dataframe(
        patient_confusion_raw
    ),
    SOURCE_DATA_DIR
    / "figure2_patient_confusion_raw.csv",
    index=True,
)

atomic_csv_save(
    confusion_dataframe(
        patient_confusion_normalized
    ),
    SOURCE_DATA_DIR
    / "figure2_patient_confusion_row_normalized.csv",
    index=True,
)

atomic_csv_save(
    confusion_dataframe(
        image_confusion_raw
    ),
    SOURCE_DATA_DIR
    / "figure2_image_confusion_raw.csv",
    index=True,
)

atomic_csv_save(
    confusion_dataframe(
        image_confusion_normalized
    ),
    SOURCE_DATA_DIR
    / "figure2_image_confusion_row_normalized.csv",
    index=True,
)


# ==================================================================================================
# 15. Per-class point estimates
# ==================================================================================================

def per_class_metrics_dataframe(
    y_true,
    y_pred,
    level_name,
):
    precision, recall, f1_values, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            zero_division=0,
        )
    )

    rows = []

    for class_id in CLASS_IDS:
        rows.append(
            {
                "level": level_name,
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "metric": "precision",
                "point_estimate": float(
                    precision[class_id]
                ),
                "support": int(
                    support[class_id]
                ),
            }
        )

        rows.append(
            {
                "level": level_name,
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "metric": "recall",
                "point_estimate": float(
                    recall[class_id]
                ),
                "support": int(
                    support[class_id]
                ),
            }
        )

        rows.append(
            {
                "level": level_name,
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "metric": "f1",
                "point_estimate": float(
                    f1_values[class_id]
                ),
                "support": int(
                    support[class_id]
                ),
            }
        )

    return pd.DataFrame(rows)


patient_per_class_point_df = (
    per_class_metrics_dataframe(
        patient_predictions_df["y_true"],
        patient_predictions_df["y_pred"],
        level_name="Patient-level",
    )
)

image_per_class_point_df = (
    per_class_metrics_dataframe(
        image_predictions_df["y_true"],
        image_predictions_df["y_pred"],
        level_name="Image-level",
    )
)


# ==================================================================================================
# 16. Load a saved RA09 patient per-class CI file when possible
#     Otherwise reproduce 5,000 stratified patient bootstraps
# ==================================================================================================

def parse_class_id(
    value,
):
    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            int,
            np.integer,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            float,
            np.floating,
        ),
    ):
        if float(value).is_integer():
            return int(value)

    match = re.search(
        r"([0-3])",
        str(value),
    )

    if match is None:
        return None

    return int(
        match.group(1)
    )


def normalize_metric_label(
    value,
):
    normalized = normalize_name(value)

    if "precision" in normalized:
        return "precision"

    if "recall" in normalized:
        return "recall"

    if "f1" in normalized and "macro" not in normalized:
        return "f1"

    return None


def try_parse_saved_per_class_ci(
    path,
):
    dataframe = safe_read_csv(path)
    dataframe = filter_to_swin_rows(dataframe)

    class_column = find_column(
        dataframe,
        aliases=[
            "class_id",
            "class_index",
            "mayo_class",
            "class",
            "label",
        ],
    )

    metric_column = find_column(
        dataframe,
        aliases=[
            "metric",
            "metric_name",
        ],
    )

    point_column = find_column(
        dataframe,
        aliases=[
            "point_estimate",
            "estimate",
            "value",
            "point",
        ],
    )

    low_column = find_column(
        dataframe,
        aliases=[
            "lower_95",
            "ci_low",
            "lower_ci",
            "lower",
            "bootstrap_ci_low",
        ],
    )

    high_column = find_column(
        dataframe,
        aliases=[
            "upper_95",
            "ci_high",
            "upper_ci",
            "upper",
            "bootstrap_ci_high",
        ],
    )

    support_column = find_column(
        dataframe,
        aliases=[
            "support",
            "n",
            "n_patients",
        ],
    )

    required = [
        class_column,
        metric_column,
        point_column,
        low_column,
        high_column,
    ]

    if any(
        value is None
        for value in required
    ):
        return None

    rows = []

    for _, row in dataframe.iterrows():
        class_id = parse_class_id(
            row[class_column]
        )

        metric_name = normalize_metric_label(
            row[metric_column]
        )

        if class_id not in CLASS_IDS:
            continue

        if metric_name is None:
            continue

        point_estimate = float(
            row[point_column]
        )

        ci_low = float(
            row[low_column]
        )

        ci_high = float(
            row[high_column]
        )

        if max(
            point_estimate,
            ci_low,
            ci_high,
        ) > 1.5:
            point_estimate /= 100.0
            ci_low /= 100.0
            ci_high /= 100.0

        if (
            support_column is not None
            and
            not pd.isna(
                row[support_column]
            )
        ):
            support = int(
                row[support_column]
            )

        else:
            support = int(
                EXPECTED_PATIENT_SUPPORT[
                    class_id
                ]
            )

        rows.append(
            {
                "level": "Patient-level",
                "class_id": class_id,
                "class_name": CLASS_NAMES[class_id],
                "metric": metric_name,
                "point_estimate": point_estimate,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "support": support,
                "ci_source": "RA09 saved bootstrap output",
            }
        )

    if len(rows) != 12:
        return None

    parsed = pd.DataFrame(rows)

    parsed = parsed.sort_values(
        [
            "metric",
            "class_id",
        ]
    ).reset_index(
        drop=True
    )

    comparison = parsed.merge(
        patient_per_class_point_df[
            [
                "class_id",
                "metric",
                "point_estimate",
            ]
        ],
        on=[
            "class_id",
            "metric",
        ],
        suffixes=(
            "_saved",
            "_recomputed",
        ),
    )

    maximum_difference = float(
        np.max(
            np.abs(
                comparison[
                    "point_estimate_saved"
                ]
                -
                comparison[
                    "point_estimate_recomputed"
                ]
            )
        )
    )

    if maximum_difference > 1e-5:
        return None

    return parsed


per_class_ci_candidates = sorted(
    [
        path
        for path in RA09.rglob("*.csv")
        if (
            "patient"
            in path.name.lower()
            and
            "class"
            in path.name.lower()
            and
            (
                "bootstrap"
                in path.name.lower()
                or
                "ci"
                in path.name.lower()
            )
        )
    ],
    key=lambda path: (
        0
        if path.name
        ==
        "official_test_patient_per_class_metrics_bootstrap_ci.csv"
        else
        1,
        path.name,
    ),
)


patient_per_class_ci_df = None
patient_per_class_ci_source = None


for candidate_path in per_class_ci_candidates:
    try:
        parsed_candidate = (
            try_parse_saved_per_class_ci(
                candidate_path
            )
        )

    except Exception:
        parsed_candidate = None

    if parsed_candidate is not None:
        patient_per_class_ci_df = (
            parsed_candidate
        )

        patient_per_class_ci_source = (
            candidate_path
        )

        register_source(
            "RA09 patient per-class bootstrap confidence intervals",
            candidate_path,
        )

        break


def bootstrap_patient_per_class_metrics(
    patient_dataframe,
    n_bootstrap,
    seed,
):
    y_true = patient_dataframe[
        "y_true"
    ].to_numpy(
        dtype=int
    )

    y_pred = patient_dataframe[
        "y_pred"
    ].to_numpy(
        dtype=int
    )

    rng = np.random.default_rng(seed)

    class_indices = {
        class_id: np.flatnonzero(
            y_true == class_id
        )
        for class_id in CLASS_IDS
    }

    bootstrap_precision = np.zeros(
        (
            n_bootstrap,
            NUM_CLASSES,
        ),
        dtype=float,
    )

    bootstrap_recall = np.zeros(
        (
            n_bootstrap,
            NUM_CLASSES,
        ),
        dtype=float,
    )

    bootstrap_f1 = np.zeros(
        (
            n_bootstrap,
            NUM_CLASSES,
        ),
        dtype=float,
    )

    for bootstrap_index in range(
        n_bootstrap
    ):
        sampled_indices = np.concatenate(
            [
                rng.choice(
                    indices,
                    size=len(indices),
                    replace=True,
                )
                for indices in class_indices.values()
            ]
        )

        precision, recall, f1_values, _ = (
            precision_recall_fscore_support(
                y_true[sampled_indices],
                y_pred[sampled_indices],
                labels=CLASS_IDS,
                zero_division=0,
            )
        )

        bootstrap_precision[
            bootstrap_index
        ] = precision

        bootstrap_recall[
            bootstrap_index
        ] = recall

        bootstrap_f1[
            bootstrap_index
        ] = f1_values

    bootstrap_values = {
        "precision": bootstrap_precision,
        "recall": bootstrap_recall,
        "f1": bootstrap_f1,
    }

    rows = []

    for metric_name, metric_array in bootstrap_values.items():
        point_subset = (
            patient_per_class_point_df[
                patient_per_class_point_df[
                    "metric"
                ]
                ==
                metric_name
            ]
            .set_index(
                "class_id"
            )
        )

        for class_id in CLASS_IDS:
            ci_low, ci_high = np.percentile(
                metric_array[:, class_id],
                [
                    2.5,
                    97.5,
                ],
            )

            rows.append(
                {
                    "level": "Patient-level",
                    "class_id": class_id,
                    "class_name": CLASS_NAMES[class_id],
                    "metric": metric_name,
                    "point_estimate": float(
                        point_subset.loc[
                            class_id,
                            "point_estimate",
                        ]
                    ),
                    "ci_low": float(ci_low),
                    "ci_high": float(ci_high),
                    "support": int(
                        EXPECTED_PATIENT_SUPPORT[
                            class_id
                        ]
                    ),
                    "ci_source": (
                        "Recomputed from locked RA09 predictions: "
                        + str(n_bootstrap)
                        + " stratified patient bootstraps; seed="
                        + str(seed)
                    ),
                }
            )

    return pd.DataFrame(
        rows
    ).sort_values(
        [
            "metric",
            "class_id",
        ]
    ).reset_index(
        drop=True
    )


if patient_per_class_ci_df is None:
    print("")
    print(
        "No parseable saved patient per-class CI file was found."
    )

    print(
        "Recomputing 5,000 stratified patient-bootstrap replicates."
    )

    patient_per_class_ci_df = (
        bootstrap_patient_per_class_metrics(
            patient_dataframe=patient_predictions_df,
            n_bootstrap=BOOTSTRAP_REPLICATES,
            seed=BOOTSTRAP_SEED,
        )
    )

    patient_per_class_ci_source = (
        "recomputed_from_locked_RA09_predictions"
    )


atomic_csv_save(
    patient_per_class_ci_df,
    SOURCE_DATA_DIR
    / "figure3_patient_per_class_metrics_with_ci.csv",
    index=False,
)

atomic_csv_save(
    patient_per_class_point_df,
    SOURCE_DATA_DIR
    / "patient_per_class_point_estimates.csv",
    index=False,
)

atomic_csv_save(
    image_per_class_point_df,
    SOURCE_DATA_DIR
    / "image_per_class_point_estimates.csv",
    index=False,
)


# ==================================================================================================
# 17. Aggregation sensitivity
# ==================================================================================================

def normalize_probability_vector(
    vector,
):
    vector = np.asarray(
        vector,
        dtype=float,
    )

    vector = np.clip(
        vector,
        0.0,
        None,
    )

    total = float(
        vector.sum()
    )

    if total <= 0:
        return np.ones(
            NUM_CLASSES,
            dtype=float,
        ) / NUM_CLASSES

    return vector / total


def majority_vote_with_probability_tie_break(
    hard_predictions,
    mean_probabilities,
):
    vote_counts = np.bincount(
        hard_predictions,
        minlength=NUM_CLASSES,
    )

    tied_classes = np.flatnonzero(
        vote_counts == vote_counts.max()
    )

    if len(tied_classes) == 1:
        return int(
            tied_classes[0]
        )

    tied_probabilities = mean_probabilities[
        tied_classes
    ]

    maximum_probability = float(
        tied_probabilities.max()
    )

    remaining_ties = tied_classes[
        np.flatnonzero(
            tied_probabilities
            ==
            maximum_probability
        )
    ]

    return int(
        remaining_ties[0]
    )


def aggregate_one_patient(
    group,
    rule,
):
    probabilities = group[
        PROBABILITY_COLUMNS
    ].to_numpy(
        dtype=float
    )

    hard_predictions = np.argmax(
        probabilities,
        axis=1,
    )

    mean_probabilities = (
        normalize_probability_vector(
            probabilities.mean(
                axis=0
            )
        )
    )

    if rule == "mean_probability":
        return int(
            np.argmax(
                mean_probabilities
            )
        )

    if rule == "median_probability":
        median_probabilities = (
            normalize_probability_vector(
                np.median(
                    probabilities,
                    axis=0,
                )
            )
        )

        return int(
            np.argmax(
                median_probabilities
            )
        )

    if rule == "majority_vote":
        return majority_vote_with_probability_tie_break(
            hard_predictions=hard_predictions,
            mean_probabilities=mean_probabilities,
        )

    if rule == "confidence_weighted_mean":
        weights = probabilities.max(
            axis=1
        )

        if float(weights.sum()) <= 0:
            weighted_probabilities = (
                mean_probabilities
            )

        else:
            weighted_probabilities = (
                normalize_probability_vector(
                    (
                        probabilities
                        * weights[:, None]
                    ).sum(
                        axis=0
                    )
                )
            )

        return int(
            np.argmax(
                weighted_probabilities
            )
        )

    if rule == "confidence_quantile_filtered_mean":
        if (
            len(group)
            <
            MIN_IMAGES_FOR_CONFIDENCE_TRIM
        ):
            filtered_probabilities = (
                mean_probabilities
            )

        else:
            confidence = probabilities.max(
                axis=1
            )

            lower_quantile = np.quantile(
                confidence,
                TRIM_FRACTION,
            )

            upper_quantile = np.quantile(
                confidence,
                1.0 - TRIM_FRACTION,
            )

            keep_mask = (
                (confidence >= lower_quantile)
                &
                (confidence <= upper_quantile)
            )

            kept_probabilities = probabilities[
                keep_mask
            ]

            if len(kept_probabilities) == 0:
                kept_probabilities = probabilities

            filtered_probabilities = (
                normalize_probability_vector(
                    kept_probabilities.mean(
                        axis=0
                    )
                )
            )

        return int(
            np.argmax(
                filtered_probabilities
            )
        )

    if rule == "maximum_predicted_class":
        return int(
            hard_predictions.max()
        )

    raise ValueError(
        "Unknown aggregation rule: "
        + str(rule)
    )


aggregation_rule_definitions = [
    (
        "mean_probability",
        "Mean probability\n(primary)",
        True,
    ),
    (
        "median_probability",
        "Median probability",
        False,
    ),
    (
        "majority_vote",
        "Majority vote",
        False,
    ),
    (
        "confidence_weighted_mean",
        "Confidence-weighted\nmean",
        False,
    ),
    (
        "confidence_quantile_filtered_mean",
        "Confidence-quantile\nfiltered mean",
        False,
    ),
    (
        "maximum_predicted_class",
        "Maximum predicted\nclass",
        False,
    ),
]


aggregation_patient_rows = []


for patient_id, group in image_predictions_df.groupby(
    "patient_id",
    sort=True,
):
    image_labels = group[
        "y_true"
    ].to_numpy(
        dtype=int
    )

    patient_row = {
        "patient_id": int(patient_id),
        "reference_modal_low": mode_low(
            image_labels
        ),
    }

    for rule_key, _, _ in aggregation_rule_definitions:
        patient_row[rule_key] = (
            aggregate_one_patient(
                group=group,
                rule=rule_key,
            )
        )

    aggregation_patient_rows.append(
        patient_row
    )


aggregation_patient_predictions_df = pd.DataFrame(
    aggregation_patient_rows
)


aggregation_summary_rows = []


for order_index, (
    rule_key,
    display_name,
    is_primary,
) in enumerate(
    aggregation_rule_definitions
):
    metrics = compute_metrics(
        aggregation_patient_predictions_df[
            "reference_modal_low"
        ],
        aggregation_patient_predictions_df[
            rule_key
        ],
    )

    aggregation_summary_rows.append(
        {
            "order": order_index,
            "rule_key": rule_key,
            "display_name": display_name,
            "is_primary": bool(is_primary),
            "reference_rule": "modal_low",
            "n_patients": int(
                len(
                    aggregation_patient_predictions_df
                )
            ),
            "qwk": metrics["qwk"],
            "mae": metrics["mae"],
            "macro_f1": metrics["macro_f1"],
            "accuracy": metrics["accuracy"],
        }
    )


aggregation_sensitivity_df = pd.DataFrame(
    aggregation_summary_rows
)


atomic_csv_save(
    aggregation_sensitivity_df,
    SOURCE_DATA_DIR
    / "figure4_aggregation_sensitivity_modal_low.csv",
    index=False,
)

atomic_csv_save(
    aggregation_patient_predictions_df,
    SOURCE_DATA_DIR
    / "figure4_aggregation_patient_predictions.csv",
    index=False,
)


# ==================================================================================================
# 18. Derived-reference sensitivity
# ==================================================================================================

reference_patient_rows = []


for patient_id, group in image_predictions_df.groupby(
    "patient_id",
    sort=True,
):
    image_labels = group[
        "y_true"
    ].to_numpy(
        dtype=int
    )

    reference_patient_rows.append(
        {
            "patient_id": int(patient_id),
            "modal_low": mode_low(
                image_labels
            ),
            "modal_high": mode_high(
                image_labels
            ),
            "median_low": median_low(
                image_labels
            ),
            "median_high": median_high(
                image_labels
            ),
            "maximum_label": int(
                image_labels.max()
            ),
        }
    )


reference_patient_dataframe = pd.DataFrame(
    reference_patient_rows
)


reference_patient_dataframe = (
    reference_patient_dataframe.merge(
        patient_predictions_df[
            [
                "patient_id",
                "y_pred",
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)


reference_rule_definitions = [
    (
        "modal_low",
        "Modal-low\n(primary)",
        True,
    ),
    (
        "modal_high",
        "Modal-high",
        False,
    ),
    (
        "median_low",
        "Median-low",
        False,
    ),
    (
        "median_high",
        "Median-high",
        False,
    ),
    (
        "maximum_label",
        "Maximum label",
        False,
    ),
]


reference_summary_rows = []


for order_index, (
    rule_key,
    display_name,
    is_primary,
) in enumerate(
    reference_rule_definitions
):
    metrics = compute_metrics(
        reference_patient_dataframe[
            rule_key
        ],
        reference_patient_dataframe[
            "y_pred"
        ],
    )

    class_counts = (
        reference_patient_dataframe[
            rule_key
        ]
        .value_counts()
        .sort_index()
        .reindex(
            CLASS_IDS,
            fill_value=0,
        )
    )

    reference_summary_rows.append(
        {
            "order": order_index,
            "rule_key": rule_key,
            "display_name": display_name,
            "is_primary": bool(is_primary),
            "prediction_rule": "mean_probability",
            "n_patients": int(
                len(
                    reference_patient_dataframe
                )
            ),
            "qwk": metrics["qwk"],
            "mae": metrics["mae"],
            "macro_f1": metrics["macro_f1"],
            "accuracy": metrics["accuracy"],
            "mayo_0": int(
                class_counts.loc[0]
            ),
            "mayo_1": int(
                class_counts.loc[1]
            ),
            "mayo_2": int(
                class_counts.loc[2]
            ),
            "mayo_3": int(
                class_counts.loc[3]
            ),
        }
    )


reference_sensitivity_df = pd.DataFrame(
    reference_summary_rows
)


atomic_csv_save(
    reference_sensitivity_df,
    SOURCE_DATA_DIR
    / "figure4_reference_sensitivity_mean_probability.csv",
    index=False,
)

atomic_csv_save(
    reference_patient_dataframe,
    SOURCE_DATA_DIR
    / "figure4_reference_patient_predictions.csv",
    index=False,
)


# ==================================================================================================
# 19. Load the RA06 uniform backbone summary
# ==================================================================================================

def choose_existing_file(
    candidates,
):
    for candidate in candidates:
        if candidate.exists():
            return candidate

    return None


BACKBONE_SUMMARY_FILE = choose_existing_file(
    [
        RA06
        / "backbone_multiseed_summary.csv",
        RA06
        / "backbone_multiseed_ranking.csv",
        RA06
        / "backbone_results_by_seed.csv",
    ]
)


def find_numeric_column(
    dataframe,
    aliases,
):
    column = find_column(
        dataframe,
        aliases=aliases,
    )

    return column


def parse_backbone_summary(
    path,
):
    dataframe = safe_read_csv(path)

    backbone_column = find_column(
        dataframe,
        aliases=[
            "backbone_name",
            "backbone",
        ],
    )

    if backbone_column is None:
        return None

    column_aliases = {
        "patient_qwk_mean": [
            "patient_qwk_mean",
            "oof_patient_qwk_mean",
        ],
        "patient_qwk_std": [
            "patient_qwk_std",
            "oof_patient_qwk_std",
        ],
        "patient_mae_mean": [
            "patient_mae_mean",
            "oof_patient_mae_mean",
        ],
        "patient_mae_std": [
            "patient_mae_std",
            "oof_patient_mae_std",
        ],
        "patient_macro_f1_mean": [
            "patient_macro_f1_mean",
            "oof_patient_macro_f1_mean",
        ],
        "patient_macro_f1_std": [
            "patient_macro_f1_std",
            "oof_patient_macro_f1_std",
        ],
        "patient_accuracy_mean": [
            "patient_accuracy_mean",
            "oof_patient_accuracy_mean",
        ],
        "patient_accuracy_std": [
            "patient_accuracy_std",
            "oof_patient_accuracy_std",
        ],
    }

    source_columns = {}

    for target_name, aliases in column_aliases.items():
        source_column = find_numeric_column(
            dataframe,
            aliases,
        )

        if source_column is None:
            return None

        source_columns[
            target_name
        ] = source_column

    output = pd.DataFrame()

    output[
        "backbone"
    ] = dataframe[
        backbone_column
    ].astype(str)

    for target_name, source_column in source_columns.items():
        output[
            target_name
        ] = pd.to_numeric(
            dataframe[source_column],
            errors="coerce",
        )

    output = output[
        output[
            "backbone"
        ].str.contains(
            "swin|vit",
            case=False,
            regex=True,
            na=False,
        )
    ].copy()

    output[
        "backbone"
    ] = output[
        "backbone"
    ].apply(
        lambda value: (
            "Swin"
            if "swin" in value.lower()
            else "ViT"
        )
    )

    output = (
        output
        .drop_duplicates(
            subset=[
                "backbone",
            ]
        )
        .set_index(
            "backbone"
        )
        .reindex(
            [
                "Swin",
                "ViT",
            ]
        )
        .reset_index()
    )

    if output.isna().any().any():
        return None

    return output


backbone_summary_df = None


if BACKBONE_SUMMARY_FILE is not None:
    try:
        backbone_summary_df = (
            parse_backbone_summary(
                BACKBONE_SUMMARY_FILE
            )
        )

    except Exception:
        backbone_summary_df = None


if backbone_summary_df is None:
    fallback_candidates = sorted(
        [
            path
            for path in RA06.rglob("*.csv")
            if (
                "backbone"
                in path.name.lower()
                and
                (
                    "summary"
                    in path.name.lower()
                    or
                    "ranking"
                    in path.name.lower()
                )
            )
        ]
    )

    for candidate_path in fallback_candidates:
        try:
            candidate_dataframe = (
                parse_backbone_summary(
                    candidate_path
                )
            )

        except Exception:
            candidate_dataframe = None

        if candidate_dataframe is not None:
            backbone_summary_df = (
                candidate_dataframe
            )

            BACKBONE_SUMMARY_FILE = (
                candidate_path
            )

            break


if backbone_summary_df is not None:
    register_source(
        "RA06 uniform three-seed backbone summary",
        BACKBONE_SUMMARY_FILE,
    )

    atomic_csv_save(
        backbone_summary_df,
        SOURCE_DATA_DIR
        / "supplementary_figureS3_backbone_summary.csv",
        index=False,
    )


# ==================================================================================================
# 20. Discover a parseable RA07 four-condition ablation summary
# ==================================================================================================

def parse_ablation_summary(
    path,
):
    dataframe = safe_read_csv(path)

    condition_column = find_column(
        dataframe,
        aliases=[
            "ablation_id",
            "condition",
            "ablation",
            "configuration",
            "experiment_name",
            "candidate_name",
        ],
    )

    if condition_column is None:
        return None

    metric_aliases = {
        "patient_qwk_mean": [
            "patient_qwk_mean",
            "oof_patient_qwk_mean",
        ],
        "patient_qwk_std": [
            "patient_qwk_std",
            "oof_patient_qwk_std",
        ],
        "patient_mae_mean": [
            "patient_mae_mean",
            "oof_patient_mae_mean",
        ],
        "patient_mae_std": [
            "patient_mae_std",
            "oof_patient_mae_std",
        ],
        "patient_macro_f1_mean": [
            "patient_macro_f1_mean",
            "oof_patient_macro_f1_mean",
        ],
        "patient_macro_f1_std": [
            "patient_macro_f1_std",
            "oof_patient_macro_f1_std",
        ],
        "patient_accuracy_mean": [
            "patient_accuracy_mean",
            "oof_patient_accuracy_mean",
        ],
        "patient_accuracy_std": [
            "patient_accuracy_std",
            "oof_patient_accuracy_std",
        ],
    }

    source_columns = {}

    for target_name, aliases in metric_aliases.items():
        source_column = find_column(
            dataframe,
            aliases=aliases,
        )

        if source_column is None:
            return None

        source_columns[
            target_name
        ] = source_column

    output = pd.DataFrame()

    output[
        "condition_raw"
    ] = dataframe[
        condition_column
    ].astype(str)

    for target_name, source_column in source_columns.items():
        output[
            target_name
        ] = pd.to_numeric(
            dataframe[source_column],
            errors="coerce",
        )

    output = output.dropna(
        subset=list(
            source_columns.keys()
        )
    ).copy()

    if len(output) != 4:
        return None

    def condition_sort_key(value):
        normalized = normalize_name(value)

        for condition_number in range(4):
            if (
                f"a{condition_number}"
                in normalized
            ):
                return condition_number

        return 99

    output[
        "condition_order"
    ] = output[
        "condition_raw"
    ].map(
        condition_sort_key
    )

    output = output.sort_values(
        [
            "condition_order",
            "condition_raw",
        ]
    ).reset_index(
        drop=True
    )

    output[
        "display_name"
    ] = output[
        "condition_raw"
    ].str.replace(
        "_",
        " ",
        regex=False,
    )

    return output


ablation_summary_df = None
ABLATION_SUMMARY_FILE = None


if RA07 is not None:
    ablation_candidates = sorted(
        [
            path
            for path in RA07.rglob("*.csv")
            if (
                "summary"
                in path.name.lower()
                or
                "ablation"
                in path.name.lower()
                or
                "multiseed"
                in path.name.lower()
            )
        ]
    )

    for candidate_path in ablation_candidates:
        try:
            candidate_dataframe = (
                parse_ablation_summary(
                    candidate_path
                )
            )

        except Exception:
            candidate_dataframe = None

        if candidate_dataframe is not None:
            ablation_summary_df = (
                candidate_dataframe
            )

            ABLATION_SUMMARY_FILE = (
                candidate_path
            )

            break


if ablation_summary_df is not None:
    register_source(
        "RA07 four-condition ablation summary",
        ABLATION_SUMMARY_FILE,
    )

    atomic_csv_save(
        ablation_summary_df,
        SOURCE_DATA_DIR
        / "supplementary_figureS4_ablation_summary.csv",
        index=False,
    )


# ==================================================================================================
# 21. Figure 1 — Route A workflow
# ==================================================================================================

def add_workflow_box(
    axis,
    x,
    y,
    width,
    height,
    title,
    subtitle,
    facecolor,
    edgecolor,
):
    box = patches.FancyBboxPatch(
        (
            x,
            y,
        ),
        width,
        height,
        boxstyle=(
            "round,pad=0.012,"
            "rounding_size=0.018"
        ),
        linewidth=1.6,
        edgecolor=edgecolor,
        facecolor=facecolor,
    )

    axis.add_patch(box)

    axis.text(
        x + width / 2,
        y + height * 0.66,
        title,
        ha="center",
        va="center",
        fontsize=10.5,
        fontweight="bold",
        color=DARK_COLOR,
    )

    axis.text(
        x + width / 2,
        y + height * 0.30,
        subtitle,
        ha="center",
        va="center",
        fontsize=8.2,
        color=DARK_COLOR,
        linespacing=1.12,
    )


def add_workflow_arrow(
    axis,
    start_x,
    start_y,
    end_x,
    end_y,
    color=DARK_COLOR,
    connectionstyle="arc3,rad=0.0",
):
    arrow = patches.FancyArrowPatch(
        (
            start_x,
            start_y,
        ),
        (
            end_x,
            end_y,
        ),
        arrowstyle="-|>",
        mutation_scale=13,
        linewidth=1.5,
        color=color,
        connectionstyle=connectionstyle,
    )

    axis.add_patch(arrow)


workflow_figure, workflow_axis = plt.subplots(
    figsize=(
        17.5,
        8.5,
    )
)

workflow_axis.set_xlim(
    0,
    1,
)

workflow_axis.set_ylim(
    0,
    1,
)

workflow_axis.axis("off")


development_boundary = patches.FancyBboxPatch(
    (
        0.025,
        0.50,
    ),
    0.95,
    0.42,
    boxstyle=(
        "round,pad=0.015,"
        "rounding_size=0.025"
    ),
    linewidth=1.7,
    edgecolor=PRIMARY_COLOR,
    facecolor="#F2F8FC",
    linestyle="--",
)

workflow_axis.add_patch(
    development_boundary
)

workflow_axis.text(
    0.04,
    0.895,
    "Development-only reconstruction: official-test objects unavailable",
    fontsize=12,
    color=PRIMARY_COLOR,
    fontweight="bold",
    ha="left",
    va="center",
)


top_nodes = [
    (
        "RA01",
        "Data audit, modal-low reference,\nand frozen patient folds",
    ),
    (
        "RA01–RA02",
        "ViT loss screening and\nthree-seed confirmation",
    ),
    (
        "RA03",
        "Development OOF\naggregation lock",
    ),
    (
        "RA04–RA05",
        "Swin loss screening and\nthree-seed confirmation",
    ),
    (
        "RA06",
        "Uniform three-seed\nSwin–ViT benchmark",
    ),
    (
        "RA07",
        "Explanatory factorial\nablation; no reselection",
    ),
]


top_x_positions = [
    0.045,
    0.201,
    0.357,
    0.513,
    0.669,
    0.825,
]

top_y = 0.63
top_width = 0.137
top_height = 0.19


for node_index, (
    title,
    subtitle,
) in enumerate(top_nodes):
    add_workflow_box(
        axis=workflow_axis,
        x=top_x_positions[node_index],
        y=top_y,
        width=top_width,
        height=top_height,
        title=title,
        subtitle=subtitle,
        facecolor="white",
        edgecolor=PRIMARY_COLOR,
    )

    if node_index < len(top_nodes) - 1:
        add_workflow_arrow(
            axis=workflow_axis,
            start_x=(
                top_x_positions[node_index]
                + top_width
            ),
            start_y=(
                top_y
                + top_height / 2
            ),
            end_x=(
                top_x_positions[
                    node_index + 1
                ]
                - 0.008
            ),
            end_y=(
                top_y
                + top_height / 2
            ),
            color=PRIMARY_COLOR,
        )


bottom_nodes = [
    (
        "RA08 — immutable pre-test lock",
        "30 exact checkpoint hashes\n"
        "Primary Swin + prespecified ViT comparator\n"
        "Reference, aggregation, TTA, endpoints, bootstrap",
        "#FFF5D9",
        ACCENT_COLOR,
    ),
    (
        "RA09 — locked official-test inference",
        "15 checkpoints per system = 3 seeds × 5 folds\n"
        "Original + horizontal flip\n"
        "Equal probability averaging",
        "#F8F2FA",
        PURPLE_COLOR,
    ),
    (
        "Grouped patient-level evaluation",
        "Mean image probabilities → patient argmax\n"
        "Primary endpoint: patient QWK\n"
        "5,000 stratified patient bootstraps",
        "#EEF8F2",
        GREEN_COLOR,
    ),
]


bottom_x_positions = [
    0.085,
    0.39,
    0.695,
]

bottom_y = 0.16
bottom_width = 0.22
bottom_height = 0.20


for node_index, (
    title,
    subtitle,
    facecolor,
    edgecolor,
) in enumerate(bottom_nodes):
    add_workflow_box(
        axis=workflow_axis,
        x=bottom_x_positions[node_index],
        y=bottom_y,
        width=bottom_width,
        height=bottom_height,
        title=title,
        subtitle=subtitle,
        facecolor=facecolor,
        edgecolor=edgecolor,
    )

    if node_index < len(bottom_nodes) - 1:
        add_workflow_arrow(
            axis=workflow_axis,
            start_x=(
                bottom_x_positions[node_index]
                + bottom_width
            ),
            start_y=(
                bottom_y
                + bottom_height / 2
            ),
            end_x=(
                bottom_x_positions[
                    node_index + 1
                ]
                - 0.012
            ),
            end_y=(
                bottom_y
                + bottom_height / 2
            ),
        )


add_workflow_arrow(
    axis=workflow_axis,
    start_x=(
        top_x_positions[-1]
        + top_width / 2
    ),
    start_y=top_y,
    end_x=(
        bottom_x_positions[0]
        + bottom_width / 2
    ),
    end_y=(
        bottom_y
        + bottom_height
    ),
    color=ACCENT_COLOR,
    connectionstyle="arc3,rad=0.28",
)


workflow_axis.text(
    0.5,
    0.955,
    "Reconstructed Route A analysis and locked official-test evaluation",
    ha="center",
    va="center",
    fontsize=16,
    fontweight="bold",
    color=DARK_COLOR,
)

workflow_axis.text(
    0.5,
    0.06,
    (
        "Legacy pre-Route-A analyses are retained for provenance only "
        "and do not contribute performance claims to Route A."
    ),
    ha="center",
    va="center",
    fontsize=9.5,
    color=NEUTRAL_COLOR,
)


workflow_figure.tight_layout()


save_figure(
    figure=workflow_figure,
    stem="Figure_1_route_A_workflow",
    description=(
        "Development-only Route A reconstruction, immutable pre-test lock, "
        "and locked official-test inference workflow."
    ),
)


workflow_source_df = pd.DataFrame(
    [
        {
            "stage_order": index + 1,
            "stage": title,
            "description": subtitle.replace(
                "\n",
                " ",
            ),
        }
        for index, (
            title,
            subtitle,
        ) in enumerate(top_nodes)
    ]
)


atomic_csv_save(
    workflow_source_df,
    SOURCE_DATA_DIR
    / "figure1_workflow_nodes.csv",
    index=False,
)


# ==================================================================================================
# 22. Figure 2 — Row-normalized confusion matrices with counts
# ==================================================================================================

def confusion_annotation_matrix(
    normalized_matrix,
    raw_matrix,
):
    annotations = np.empty(
        normalized_matrix.shape,
        dtype=object,
    )

    for row_index in range(
        normalized_matrix.shape[0]
    ):
        for column_index in range(
            normalized_matrix.shape[1]
        ):
            annotations[
                row_index,
                column_index,
            ] = (
                f"{100.0 * normalized_matrix[row_index, column_index]:.1f}%\n"
                f"(n={int(raw_matrix[row_index, column_index])})"
            )

    return annotations


patient_annotations = (
    confusion_annotation_matrix(
        patient_confusion_normalized,
        patient_confusion_raw,
    )
)

image_annotations = (
    confusion_annotation_matrix(
        image_confusion_normalized,
        image_confusion_raw,
    )
)


patient_tick_labels = [
    (
        f"{CLASS_NAMES[class_id]}\n"
        f"(n={EXPECTED_PATIENT_SUPPORT[class_id]})"
    )
    for class_id in CLASS_IDS
]


image_support = (
    image_predictions_df[
        "y_true"
    ]
    .value_counts()
    .sort_index()
    .reindex(
        CLASS_IDS,
        fill_value=0,
    )
)


image_tick_labels = [
    (
        f"{CLASS_NAMES[class_id]}\n"
        f"(n={int(image_support.loc[class_id])})"
    )
    for class_id in CLASS_IDS
]


confusion_figure = plt.figure(
    figsize=(
        14.5,
        6.2,
    )
)

confusion_grid = confusion_figure.add_gridspec(
    1,
    3,
    width_ratios=[
        1,
        1,
        0.045,
    ],
    wspace=0.34,
)

patient_axis = confusion_figure.add_subplot(
    confusion_grid[0, 0]
)

image_axis = confusion_figure.add_subplot(
    confusion_grid[0, 1]
)

colorbar_axis = confusion_figure.add_subplot(
    confusion_grid[0, 2]
)


sns.heatmap(
    patient_confusion_normalized,
    annot=patient_annotations,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.7,
    linecolor="white",
    cbar=False,
    xticklabels=[
        CLASS_NAMES[class_id]
        for class_id in CLASS_IDS
    ],
    yticklabels=patient_tick_labels,
    annot_kws={
        "fontsize": 10,
    },
    ax=patient_axis,
)


sns.heatmap(
    image_confusion_normalized,
    annot=image_annotations,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.7,
    linecolor="white",
    cbar=True,
    cbar_ax=colorbar_axis,
    xticklabels=[
        CLASS_NAMES[class_id]
        for class_id in CLASS_IDS
    ],
    yticklabels=image_tick_labels,
    annot_kws={
        "fontsize": 10,
    },
    ax=image_axis,
)


patient_axis.set_title(
    "(a) Patient level",
    fontweight="bold",
    pad=12,
)

patient_axis.set_xlabel(
    "Predicted MES"
)

patient_axis.set_ylabel(
    "Derived modal-low patient reference"
)

patient_axis.set_xticklabels(
    patient_axis.get_xticklabels(),
    rotation=0,
)

patient_axis.set_yticklabels(
    patient_axis.get_yticklabels(),
    rotation=0,
)


image_axis.set_title(
    "(b) Image level",
    fontweight="bold",
    pad=12,
)

image_axis.set_xlabel(
    "Predicted MES"
)

image_axis.set_ylabel(
    "Image-level MES label"
)

image_axis.set_xticklabels(
    image_axis.get_xticklabels(),
    rotation=0,
)

image_axis.set_yticklabels(
    image_axis.get_yticklabels(),
    rotation=0,
)


colorbar_axis.set_ylabel(
    "Row proportion"
)


confusion_figure.suptitle(
    "Locked Swin official-test confusion matrices",
    fontsize=15,
    fontweight="bold",
    y=1.01,
)

confusion_figure.subplots_adjust(
    top=0.86,
)


save_figure(
    figure=confusion_figure,
    stem="Figure_2_locked_swin_confusion_matrices",
    description=(
        "Locked Swin patient- and image-level row-normalized "
        "official-test confusion matrices with raw counts."
    ),
)


# ==================================================================================================
# 23. Figure 3 — Patient-level per-class performance with confidence intervals
# ==================================================================================================

metric_display_names = {
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1 score",
}


per_class_figure, per_class_axes = plt.subplots(
    1,
    3,
    figsize=(
        15.5,
        5.2,
    ),
    sharey=True,
)


x_positions = np.arange(
    NUM_CLASSES
)


x_labels = [
    (
        f"{CLASS_NAMES[class_id]}\n"
        f"(n={EXPECTED_PATIENT_SUPPORT[class_id]})"
    )
    for class_id in CLASS_IDS
]


for axis_index, metric_name in enumerate(
    [
        "precision",
        "recall",
        "f1",
    ]
):
    axis = per_class_axes[
        axis_index
    ]

    metric_dataframe = (
        patient_per_class_ci_df[
            patient_per_class_ci_df[
                "metric"
            ]
            ==
            metric_name
        ]
        .sort_values(
            "class_id"
        )
        .reset_index(
            drop=True
        )
    )

    point_values = metric_dataframe[
        "point_estimate"
    ].to_numpy(
        dtype=float
    )

    ci_low_values = metric_dataframe[
        "ci_low"
    ].to_numpy(
        dtype=float
    )

    ci_high_values = metric_dataframe[
        "ci_high"
    ].to_numpy(
        dtype=float
    )

    lower_errors = (
        point_values
        - ci_low_values
    )

    upper_errors = (
        ci_high_values
        - point_values
    )

    bars = axis.bar(
        x_positions,
        point_values,
        color=PRIMARY_COLOR,
        edgecolor="white",
        linewidth=0.8,
        width=0.68,
        zorder=2,
    )

    axis.errorbar(
        x_positions,
        point_values,
        yerr=np.vstack(
            [
                lower_errors,
                upper_errors,
            ]
        ),
        fmt="none",
        ecolor=DARK_COLOR,
        elinewidth=1.3,
        capsize=4,
        capthick=1.3,
        zorder=3,
    )

    for bar, point_value in zip(
        bars,
        point_values,
    ):
        axis.text(
            bar.get_x()
            + bar.get_width() / 2,
            min(
                1.025,
                point_value + 0.035,
            ),
            f"{point_value:.2f}",
            ha="center",
            va="bottom",
            fontsize=9,
            color=DARK_COLOR,
        )

    axis.set_title(
        metric_display_names[
            metric_name
        ],
        fontweight="bold",
    )

    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        x_labels,
        rotation=0,
    )

    axis.set_ylim(
        0,
        1.08,
    )

    axis.set_xlabel(
        "Derived modal-low patient reference"
    )

    if axis_index == 0:
        axis.set_ylabel(
            "Score"
        )


per_class_figure.suptitle(
    (
        "Locked Swin patient-level class performance "
        "with 95% bootstrap confidence intervals"
    ),
    fontsize=15,
    fontweight="bold",
    y=1.04,
)

per_class_figure.text(
    0.5,
    -0.01,
    (
        "Confidence intervals use stratified patient resampling. "
        "Mayo 2 and Mayo 3 estimates are based on only 5 and 6 patients."
    ),
    ha="center",
    va="top",
    fontsize=9.5,
    color=NEUTRAL_COLOR,
)

per_class_figure.tight_layout()


save_figure(
    figure=per_class_figure,
    stem="Figure_3_patient_per_class_metrics_with_ci",
    description=(
        "Locked Swin patient-level precision, recall, and F1 "
        "with exact support and 95% stratified patient-bootstrap CIs."
    ),
)


# ==================================================================================================
# 24. Figure 4 — Aggregation and reference sensitivity
# ==================================================================================================

def draw_horizontal_metric_panel(
    axis,
    dataframe,
    metric,
    title,
    lower_is_better,
):
    dataframe = dataframe.sort_values(
        "order"
    ).reset_index(
        drop=True
    )

    y_positions = np.arange(
        len(dataframe)
    )

    colors = [
        (
            ACCENT_COLOR
            if is_primary
            else
            SECONDARY_COLOR
        )
        for is_primary in dataframe[
            "is_primary"
        ]
    ]

    values = dataframe[
        metric
    ].to_numpy(
        dtype=float
    )

    bars = axis.barh(
        y_positions,
        values,
        color=colors,
        edgecolor="white",
        linewidth=0.8,
        height=0.66,
    )

    axis.set_yticks(
        y_positions
    )

    axis.set_yticklabels(
        dataframe[
            "display_name"
        ].tolist()
    )

    axis.invert_yaxis()

    axis.set_title(
        title,
        fontweight="bold",
        pad=10,
    )

    if lower_is_better:
        axis.set_xlabel(
            "MAE (lower is better)"
        )

    else:
        axis.set_xlabel(
            "QWK (higher is better)"
        )

    primary_values = dataframe.loc[
        dataframe[
            "is_primary"
        ],
        metric,
    ]

    if len(primary_values) == 1:
        axis.axvline(
            float(
                primary_values.iloc[0]
            ),
            color=ACCENT_COLOR,
            linestyle="--",
            linewidth=1.2,
        )

    value_range = max(
        0.10,
        float(values.max() - values.min()),
    )

    for bar, value in zip(
        bars,
        values,
    ):
        axis.text(
            value + value_range * 0.03,
            bar.get_y()
            + bar.get_height() / 2,
            f"{value:.3f}",
            ha="left",
            va="center",
            fontsize=9,
        )

    if metric == "qwk":
        minimum_limit = min(
            0.0,
            float(values.min()) - 0.06,
        )

        maximum_limit = max(
            1.0,
            float(values.max()) + 0.12,
        )

    else:
        minimum_limit = 0.0
        maximum_limit = (
            float(values.max()) * 1.20
            + 0.03
        )

    axis.set_xlim(
        minimum_limit,
        maximum_limit,
    )


sensitivity_figure, sensitivity_axes = plt.subplots(
    2,
    2,
    figsize=(
        15.5,
        9.0,
    )
)


draw_horizontal_metric_panel(
    axis=sensitivity_axes[0, 0],
    dataframe=aggregation_sensitivity_df,
    metric="qwk",
    title="(a) Aggregation sensitivity: patient QWK",
    lower_is_better=False,
)

draw_horizontal_metric_panel(
    axis=sensitivity_axes[1, 0],
    dataframe=aggregation_sensitivity_df,
    metric="mae",
    title="(b) Aggregation sensitivity: patient MAE",
    lower_is_better=True,
)

draw_horizontal_metric_panel(
    axis=sensitivity_axes[0, 1],
    dataframe=reference_sensitivity_df,
    metric="qwk",
    title="(c) Derived-reference sensitivity: patient QWK",
    lower_is_better=False,
)

draw_horizontal_metric_panel(
    axis=sensitivity_axes[1, 1],
    dataframe=reference_sensitivity_df,
    metric="mae",
    title="(d) Derived-reference sensitivity: patient MAE",
    lower_is_better=True,
)


legend_handles = [
    patches.Patch(
        facecolor=ACCENT_COLOR,
        edgecolor="white",
        label="Prespecified primary rule",
    ),
    patches.Patch(
        facecolor=SECONDARY_COLOR,
        edgecolor="white",
        label="Descriptive sensitivity rule",
    ),
]


sensitivity_figure.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(
        0.5,
        1.005,
    ),
)


sensitivity_figure.suptitle(
    (
        "Locked Swin official-test aggregation "
        "and derived-reference sensitivity"
    ),
    fontsize=15,
    fontweight="bold",
    y=1.055,
)


sensitivity_figure.text(
    0.5,
    -0.01,
    (
        "Rules are displayed in a prespecified conceptual order, "
        "not sorted by official-test performance. "
        "Sensitivity results do not replace the locked primary analysis."
    ),
    ha="center",
    va="top",
    fontsize=9.5,
    color=NEUTRAL_COLOR,
)


sensitivity_figure.tight_layout(
    rect=[
        0,
        0.02,
        1,
        0.96,
    ]
)


save_figure(
    figure=sensitivity_figure,
    stem="Figure_4_aggregation_and_reference_sensitivity",
    description=(
        "Locked Swin patient-level QWK and MAE under alternative "
        "prediction-aggregation and derived-reference rules."
    ),
)


# ==================================================================================================
# 25. Supplementary Figure S1 — Raw-count confusion matrices
# ==================================================================================================

raw_confusion_figure = plt.figure(
    figsize=(
        14.5,
        5.9,
    )
)

raw_grid = raw_confusion_figure.add_gridspec(
    1,
    4,
    width_ratios=[
        1,
        0.045,
        1,
        0.045,
    ],
    wspace=0.35,
)

raw_patient_axis = raw_confusion_figure.add_subplot(
    raw_grid[0, 0]
)

raw_patient_colorbar = raw_confusion_figure.add_subplot(
    raw_grid[0, 1]
)

raw_image_axis = raw_confusion_figure.add_subplot(
    raw_grid[0, 2]
)

raw_image_colorbar = raw_confusion_figure.add_subplot(
    raw_grid[0, 3]
)


sns.heatmap(
    patient_confusion_raw,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    linewidths=0.7,
    linecolor="white",
    xticklabels=[
        CLASS_NAMES[class_id]
        for class_id in CLASS_IDS
    ],
    yticklabels=patient_tick_labels,
    cbar=True,
    cbar_ax=raw_patient_colorbar,
    ax=raw_patient_axis,
)


sns.heatmap(
    image_confusion_raw,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    linewidths=0.7,
    linecolor="white",
    xticklabels=[
        CLASS_NAMES[class_id]
        for class_id in CLASS_IDS
    ],
    yticklabels=image_tick_labels,
    cbar=True,
    cbar_ax=raw_image_colorbar,
    ax=raw_image_axis,
)


raw_patient_axis.set_title(
    "(a) Patient level",
    fontweight="bold",
)

raw_patient_axis.set_xlabel(
    "Predicted MES"
)

raw_patient_axis.set_ylabel(
    "Derived modal-low patient reference"
)

raw_patient_axis.set_xticklabels(
    raw_patient_axis.get_xticklabels(),
    rotation=0,
)

raw_patient_axis.set_yticklabels(
    raw_patient_axis.get_yticklabels(),
    rotation=0,
)


raw_image_axis.set_title(
    "(b) Image level",
    fontweight="bold",
)

raw_image_axis.set_xlabel(
    "Predicted MES"
)

raw_image_axis.set_ylabel(
    "Image-level MES label"
)

raw_image_axis.set_xticklabels(
    raw_image_axis.get_xticklabels(),
    rotation=0,
)

raw_image_axis.set_yticklabels(
    raw_image_axis.get_yticklabels(),
    rotation=0,
)


raw_patient_colorbar.set_ylabel(
    "Count"
)

raw_image_colorbar.set_ylabel(
    "Count"
)


raw_confusion_figure.suptitle(
    (
        "Supplementary Figure S1. "
        "Locked Swin official-test confusion counts"
    ),
    fontsize=15,
    fontweight="bold",
    y=1.02,
)


raw_confusion_figure.tight_layout()


save_figure(
    figure=raw_confusion_figure,
    stem="Supplementary_Figure_S1_confusion_counts",
    description=(
        "Raw-count patient- and image-level confusion matrices "
        "for the locked Swin official-test predictions."
    ),
)


# ==================================================================================================
# 26. Supplementary Figure S2 — Image versus patient per-class metrics
# ==================================================================================================

combined_per_class_df = pd.concat(
    [
        image_per_class_point_df,
        patient_per_class_point_df,
    ],
    ignore_index=True,
)


atomic_csv_save(
    combined_per_class_df,
    SOURCE_DATA_DIR
    / "supplementary_figureS2_image_vs_patient_per_class.csv",
    index=False,
)


supplementary_per_class_figure, supplementary_per_class_axes = plt.subplots(
    1,
    3,
    figsize=(
        15.5,
        5.1,
    ),
    sharey=True,
)


for axis_index, metric_name in enumerate(
    [
        "precision",
        "recall",
        "f1",
    ]
):
    axis = supplementary_per_class_axes[
        axis_index
    ]

    metric_dataframe = combined_per_class_df[
        combined_per_class_df[
            "metric"
        ]
        ==
        metric_name
    ].copy()

    image_values = (
        metric_dataframe[
            metric_dataframe[
                "level"
            ]
            ==
            "Image-level"
        ]
        .sort_values(
            "class_id"
        )[
            "point_estimate"
        ]
        .to_numpy(
            dtype=float
        )
    )

    patient_values = (
        metric_dataframe[
            metric_dataframe[
                "level"
            ]
            ==
            "Patient-level"
        ]
        .sort_values(
            "class_id"
        )[
            "point_estimate"
        ]
        .to_numpy(
            dtype=float
        )
    )

    bar_width = 0.36

    axis.bar(
        x_positions - bar_width / 2,
        image_values,
        width=bar_width,
        color=NEUTRAL_COLOR,
        label="Image level",
    )

    axis.bar(
        x_positions + bar_width / 2,
        patient_values,
        width=bar_width,
        color=PRIMARY_COLOR,
        label="Patient level",
    )

    axis.set_title(
        metric_display_names[
            metric_name
        ],
        fontweight="bold",
    )

    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        [
            CLASS_NAMES[class_id]
            for class_id in CLASS_IDS
        ]
    )

    axis.set_ylim(
        0,
        1.05,
    )

    axis.set_xlabel(
        "MES class"
    )

    if axis_index == 0:
        axis.set_ylabel(
            "Point estimate"
        )


legend_handles, legend_labels = (
    supplementary_per_class_axes[
        0
    ].get_legend_handles_labels()
)


supplementary_per_class_figure.legend(
    legend_handles,
    legend_labels,
    loc="upper center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(
        0.5,
        1.01,
    ),
)


supplementary_per_class_figure.suptitle(
    (
        "Supplementary Figure S2. "
        "Image- and patient-level per-class point estimates"
    ),
    fontsize=15,
    fontweight="bold",
    y=1.07,
)


supplementary_per_class_figure.text(
    0.5,
    -0.01,
    (
        "Image-level estimates are descriptive and do not account "
        "for within-patient dependence. Patient-level uncertainty "
        "is shown in main Figure 3."
    ),
    ha="center",
    va="top",
    fontsize=9.5,
    color=NEUTRAL_COLOR,
)


supplementary_per_class_figure.tight_layout()


save_figure(
    figure=supplementary_per_class_figure,
    stem="Supplementary_Figure_S2_image_vs_patient_per_class",
    description=(
        "Descriptive image- and patient-level per-class precision, "
        "recall, and F1 point estimates."
    ),
)


# ==================================================================================================
# 27. Supplementary Figure S3 — Uniform Swin-versus-ViT development benchmark
# ==================================================================================================

if backbone_summary_df is not None:
    benchmark_figure, benchmark_axes = plt.subplots(
        1,
        4,
        figsize=(
            15.5,
            4.8,
        )
    )

    benchmark_metric_specifications = [
        (
            "patient_qwk_mean",
            "patient_qwk_std",
            "Patient QWK",
            False,
        ),
        (
            "patient_mae_mean",
            "patient_mae_std",
            "Patient MAE",
            True,
        ),
        (
            "patient_macro_f1_mean",
            "patient_macro_f1_std",
            "Patient macro F1",
            False,
        ),
        (
            "patient_accuracy_mean",
            "patient_accuracy_std",
            "Patient accuracy",
            False,
        ),
    ]

    benchmark_x_positions = np.arange(
        len(
            backbone_summary_df
        )
    )

    benchmark_colors = [
        PRIMARY_COLOR,
        COMPARATOR_COLOR,
    ]

    for axis, specification in zip(
        benchmark_axes,
        benchmark_metric_specifications,
    ):
        mean_column = specification[0]
        std_column = specification[1]
        title = specification[2]
        lower_is_better = specification[3]

        mean_values = backbone_summary_df[
            mean_column
        ].to_numpy(
            dtype=float
        )

        standard_deviations = backbone_summary_df[
            std_column
        ].to_numpy(
            dtype=float
        )

        bars = axis.bar(
            benchmark_x_positions,
            mean_values,
            color=benchmark_colors,
            edgecolor="white",
            width=0.64,
        )

        axis.errorbar(
            benchmark_x_positions,
            mean_values,
            yerr=standard_deviations,
            fmt="none",
            ecolor=DARK_COLOR,
            capsize=4,
            linewidth=1.2,
        )

        for bar, value in zip(
            bars,
            mean_values,
        ):
            axis.text(
                bar.get_x()
                + bar.get_width() / 2,
                value
                + max(
                    0.006,
                    float(mean_values.max()) * 0.015,
                ),
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

        axis.set_xticks(
            benchmark_x_positions
        )

        axis.set_xticklabels(
            backbone_summary_df[
                "backbone"
            ]
        )

        axis.set_title(
            title,
            fontweight="bold",
        )

        if lower_is_better:
            axis.set_ylabel(
                "Lower is better"
            )

        else:
            axis.set_ylabel(
                "Higher is better"
            )

        if mean_column == "patient_mae_mean":
            axis.set_ylim(
                0,
                float(
                    np.max(
                        mean_values
                        + standard_deviations
                    )
                )
                + 0.08,
            )

        else:
            lower_limit = max(
                0.0,
                float(
                    np.min(
                        mean_values
                        - standard_deviations
                    )
                )
                - 0.06,
            )

            upper_limit = min(
                1.05,
                float(
                    np.max(
                        mean_values
                        + standard_deviations
                    )
                )
                + 0.08,
            )

            axis.set_ylim(
                lower_limit,
                upper_limit,
            )

    benchmark_figure.suptitle(
        (
            "Supplementary Figure S3. "
            "Uniform three-seed development benchmark"
        ),
        fontsize=15,
        fontweight="bold",
        y=1.03,
    )

    benchmark_figure.text(
        0.5,
        -0.01,
        (
            "Bars show mean pooled out-of-fold patient performance "
            "across seeds 42, 2025, and 2026; error bars show "
            "standard deviation across seeds."
        ),
        ha="center",
        va="top",
        fontsize=9.5,
        color=NEUTRAL_COLOR,
    )

    benchmark_figure.tight_layout()


    save_figure(
        figure=benchmark_figure,
        stem="Supplementary_Figure_S3_uniform_backbone_benchmark",
        description=(
            "Uniform three-seed Route A development benchmark "
            "comparing the representative Swin and ViT systems."
        ),
    )

else:
    print("")
    print(
        "WARNING: No parseable RA06 backbone summary was found. "
        "Supplementary Figure S3 was skipped."
    )


# ==================================================================================================
# 28. Supplementary Figure S4 — Route A factorial ablation
# ==================================================================================================

if ablation_summary_df is not None:
    ablation_figure, ablation_axes = plt.subplots(
        2,
        2,
        figsize=(
            13.5,
            8.2,
        )
    )

    ablation_metric_specifications = [
        (
            "patient_qwk_mean",
            "patient_qwk_std",
            "Patient QWK",
            False,
        ),
        (
            "patient_mae_mean",
            "patient_mae_std",
            "Patient MAE",
            True,
        ),
        (
            "patient_macro_f1_mean",
            "patient_macro_f1_std",
            "Patient macro F1",
            False,
        ),
        (
            "patient_accuracy_mean",
            "patient_accuracy_std",
            "Patient accuracy",
            False,
        ),
    ]

    ablation_x_positions = np.arange(
        len(
            ablation_summary_df
        )
    )

    ablation_colors = [
        PRIMARY_COLOR,
        SECONDARY_COLOR,
        PURPLE_COLOR,
        NEUTRAL_COLOR,
    ]

    for axis, specification in zip(
        ablation_axes.ravel(),
        ablation_metric_specifications,
    ):
        mean_column = specification[0]
        std_column = specification[1]
        title = specification[2]
        lower_is_better = specification[3]

        mean_values = ablation_summary_df[
            mean_column
        ].to_numpy(
            dtype=float
        )

        standard_deviations = ablation_summary_df[
            std_column
        ].to_numpy(
            dtype=float
        )

        bars = axis.bar(
            ablation_x_positions,
            mean_values,
            color=ablation_colors,
            edgecolor="white",
            width=0.68,
        )

        axis.errorbar(
            ablation_x_positions,
            mean_values,
            yerr=standard_deviations,
            fmt="none",
            ecolor=DARK_COLOR,
            capsize=4,
            linewidth=1.2,
        )

        for bar, value in zip(
            bars,
            mean_values,
        ):
            axis.text(
                bar.get_x()
                + bar.get_width() / 2,
                value
                + max(
                    0.006,
                    float(mean_values.max()) * 0.012,
                ),
                f"{value:.3f}",
                ha="center",
                va="bottom",
                fontsize=8.5,
            )

        axis.set_xticks(
            ablation_x_positions
        )

        axis.set_xticklabels(
            ablation_summary_df[
                "display_name"
            ],
            rotation=15,
            ha="right",
        )

        axis.set_title(
            title,
            fontweight="bold",
        )

        if lower_is_better:
            axis.set_ylabel(
                "Lower is better"
            )

        else:
            axis.set_ylabel(
                "Higher is better"
            )

        if mean_column == "patient_mae_mean":
            axis.set_ylim(
                0,
                float(
                    np.max(
                        mean_values
                        + standard_deviations
                    )
                )
                + 0.08,
            )

        else:
            axis.set_ylim(
                max(
                    0.0,
                    float(
                        np.min(
                            mean_values
                            - standard_deviations
                        )
                    )
                    - 0.08,
                ),
                min(
                    1.05,
                    float(
                        np.max(
                            mean_values
                            + standard_deviations
                        )
                    )
                    + 0.08,
                ),
            )

    ablation_figure.suptitle(
        (
            "Supplementary Figure S4. "
            "Locked explanatory Swin factorial ablation"
        ),
        fontsize=15,
        fontweight="bold",
        y=1.02,
    )

    ablation_figure.text(
        0.5,
        -0.01,
        (
            "Bars show three-seed mean pooled out-of-fold patient "
            "performance; error bars show standard deviation. "
            "The ablation could not replace the locked primary model."
        ),
        ha="center",
        va="top",
        fontsize=9.5,
        color=NEUTRAL_COLOR,
    )

    ablation_figure.tight_layout()


    save_figure(
        figure=ablation_figure,
        stem="Supplementary_Figure_S4_locked_factorial_ablation",
        description=(
            "Four-condition explanatory Route A Swin factorial ablation."
        ),
    )

else:
    print("")
    print(
        "WARNING: No parseable four-condition RA07 ablation summary "
        "was found. Supplementary Figure S4 was skipped."
    )


# ==================================================================================================
# 29. Draft figure captions
# ==================================================================================================

caption_lines = [
    "FIGURE CAPTIONS — ROUTE A MANUSCRIPT",
    "====================================",
    "",
    (
        "Figure 1. Reconstructed Route A analysis and locked official-test "
        "evaluation. The official LIMUC development partition was used to "
        "audit the derived modal-low patient reference, freeze five "
        "patient-grouped folds, screen and confirm ViT and Swin losses, "
        "lock patient aggregation from development out-of-fold predictions, "
        "and perform a uniform three-seed backbone comparison. The exact "
        "checkpoint ensemble, preprocessing, test-time augmentation, "
        "aggregation, patient reference, endpoints, and bootstrap procedure "
        "were frozen before official-test re-evaluation."
    ),
    "",
    (
        "Figure 2. Locked Swin official-test confusion matrices. "
        "Row-normalized patient- and image-level confusion matrices are "
        "shown with raw cell counts. Patient-level results use the modal "
        "image-level MES within each patient, with lower-class resolution "
        "of modal ties, as the derived reference. The patient analysis "
        "contains 86 patients, including 56 Mayo 0, 19 Mayo 1, 5 Mayo 2, "
        "and 6 Mayo 3 patients. The image analysis contains 1,686 images."
    ),
    "",
    (
        "Figure 3. Locked Swin patient-level class performance. "
        "Precision, recall, and F1 are shown for each derived modal-low "
        "MES class. Error bars indicate 95% confidence intervals obtained "
        "through stratified patient-bootstrap resampling. Exact support is "
        "shown below each class. Mayo 2 and Mayo 3 estimates are based on "
        "only five and six patients and should be regarded as imprecise "
        "and hypothesis-generating."
    ),
    "",
    (
        "Figure 4. Prediction-aggregation and derived-reference sensitivity "
        "of the locked Swin system. Patient QWK and MAE are shown under "
        "alternative aggregation rules evaluated against the modal-low "
        "reference and alternative derived references evaluated using the "
        "locked mean-probability prediction. The prespecified primary rules "
        "are highlighted. Sensitivity analyses did not replace the locked "
        "primary analysis."
    ),
    "",
    (
        "Supplementary Figure S1. Raw-count patient- and image-level "
        "official-test confusion matrices for the locked Swin system."
    ),
    "",
    (
        "Supplementary Figure S2. Image- and patient-level per-class "
        "precision, recall, and F1 point estimates. Image-level values are "
        "descriptive and do not account for within-patient dependence."
    ),
    "",
    (
        "Supplementary Figure S3. Uniform three-seed development benchmark. "
        "Bars show mean pooled out-of-fold patient performance for the "
        "representative Swin and ViT systems, and error bars show standard "
        "deviation across seeds 42, 2025, and 2026."
    ),
    "",
    (
        "Supplementary Figure S4. Explanatory Route A factorial ablation. "
        "Bars show three-seed mean pooled out-of-fold patient performance "
        "for the four prespecified ablation conditions. The ablation was "
        "not permitted to replace the locked primary configuration."
    ),
]


caption_text = "\n".join(
    caption_lines
)


atomic_text_save(
    caption_text + "\n",
    OUTPUT_ROOT
    / "figure_captions_draft.txt",
)


# ==================================================================================================
# 30. Source and output manifests
# ==================================================================================================

source_manifest_rows = []


for source_label, source_path in sorted(
    USED_SOURCE_FILES.items(),
    key=lambda item: item[0],
):
    source_path = Path(source_path)

    source_manifest_rows.append(
        {
            "source_label": source_label,
            "path": str(source_path),
            "size_bytes": int(
                source_path.stat().st_size
            ),
            "sha256": sha256_file(
                source_path
            ),
        }
    )


source_manifest_df = pd.DataFrame(
    source_manifest_rows
)


atomic_csv_save(
    source_manifest_df,
    MANIFEST_DIR
    / "figure_source_file_manifest.csv",
    index=False,
)


figure_manifest_df = pd.DataFrame(
    GENERATED_FIGURES
)


if not figure_manifest_df.empty:
    figure_manifest_df[
        "size_bytes"
    ] = figure_manifest_df[
        "path"
    ].map(
        lambda value: int(
            Path(value).stat().st_size
        )
    )

    figure_manifest_df[
        "sha256"
    ] = figure_manifest_df[
        "path"
    ].map(
        lambda value: sha256_file(
            Path(value)
        )
    )


atomic_csv_save(
    figure_manifest_df,
    MANIFEST_DIR
    / "generated_figure_manifest.csv",
    index=False,
)


generation_manifest = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook_name": (
        "10_route_A_manuscript_figures.ipynb"
    ),
    "revision_root": str(
        REVISION_ROOT
    ),
    "output_root": str(
        OUTPUT_ROOT
    ),
    "route_A_directories": {
        "RA01": str(RA01),
        "RA03": str(RA03),
        "RA06": str(RA06),
        "RA07": (
            str(RA07)
            if RA07 is not None
            else None
        ),
        "RA08": str(RA08),
        "RA09": str(RA09),
    },
    "safety": {
        "training_performed": False,
        "checkpoints_loaded": False,
        "raw_images_accessed": False,
        "model_inference_performed": False,
        "model_selection_performed": False,
        "legacy_results_used": False,
    },
    "pretest_lock": {
        "expected_fingerprint": (
            EXPECTED_LOCK_FINGERPRINT
        ),
        "fingerprint_validated": (
            LOCK_FINGERPRINT_VALIDATED
        ),
        "fingerprint_source": (
            str(LOCK_FINGERPRINT_SOURCE)
            if LOCK_FINGERPRINT_SOURCE
            is not None
            else None
        ),
    },
    "official_test_validation": {
        "image_prediction_source": str(
            IMAGE_PREDICTION_FILE
        ),
        "n_images": int(
            len(image_predictions_df)
        ),
        "n_patients": int(
            len(patient_predictions_df)
        ),
        "patient_support": {
            str(key): int(value)
            for key, value in (
                observed_patient_support.items()
            )
        },
        "image_metrics": image_metrics,
        "patient_metrics": patient_metrics,
    },
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "stratification": (
            "derived modal-low patient reference"
        ),
        "ci_source": str(
            patient_per_class_ci_source
        ),
    },
    "generated_figure_count": int(
        figure_manifest_df[
            "figure_stem"
        ].nunique()
    )
    if not figure_manifest_df.empty
    else 0,
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib": mpl.__version__,
        "seaborn": sns.__version__,
    },
}


atomic_json_save(
    generation_manifest,
    MANIFEST_DIR
    / "figure_generation_manifest.json",
)


# ==================================================================================================
# 31. Syntax-safe final summary
# ==================================================================================================

if not figure_manifest_df.empty:
    generated_figure_stems = sorted(
        figure_manifest_df[
            "figure_stem"
        ]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

else:
    generated_figure_stems = []


if generated_figure_stems:
    generated_figure_text = "\n".join(
        [
            "- " + stem
            for stem in generated_figure_stems
        ]
    )

else:
    generated_figure_text = "- None"


summary_lines = [
    "10_route_A_manuscript_figures COMPLETED",
    "=======================================",
    "",
    "Created UTC:",
    str(
        generation_manifest[
            "created_utc"
        ]
    ),
    "",
    "Revision root:",
    str(REVISION_ROOT),
    "",
    "Output root:",
    str(OUTPUT_ROOT),
    "",
    "Training performed:",
    "NO",
    "",
    "Checkpoints loaded:",
    "NO",
    "",
    "Raw LIMUC images accessed:",
    "NO",
    "",
    "Model inference performed:",
    "NO",
    "",
    "Model selection performed:",
    "NO",
    "",
    "Legacy results used:",
    "NO",
    "",
    "Locked official-test image source:",
    str(IMAGE_PREDICTION_FILE),
    "",
    "Official-test images:",
    str(
        len(image_predictions_df)
    ),
    "",
    "Official-test patients:",
    str(
        len(patient_predictions_df)
    ),
    "",
    "Modal-low patient support:",
    json.dumps(
        {
            str(key): int(value)
            for key, value in (
                observed_patient_support.items()
            )
        },
        indent=2,
    ),
    "",
    "Validated Swin image metrics:",
    json.dumps(
        image_metrics,
        indent=2,
    ),
    "",
    "Validated Swin patient metrics:",
    json.dumps(
        patient_metrics,
        indent=2,
    ),
    "",
    "Patient per-class CI source:",
    str(patient_per_class_ci_source),
    "",
    "Generated figure stems:",
    generated_figure_text,
    "",
    "PNG directory:",
    str(PNG_DIR),
    "",
    "SVG directory:",
    str(SVG_DIR),
    "",
    "PDF directory:",
    str(PDF_DIR),
    "",
    "Source-data directory:",
    str(SOURCE_DATA_DIR),
    "",
    "Figure manifest:",
    str(
        MANIFEST_DIR
        / "generated_figure_manifest.csv"
    ),
    "",
    "Source-file manifest:",
    str(
        MANIFEST_DIR
        / "figure_source_file_manifest.csv"
    ),
    "",
    "Generation manifest:",
    str(
        MANIFEST_DIR
        / "figure_generation_manifest.json"
    ),
    "",
    "Draft figure captions:",
    str(
        OUTPUT_ROOT
        / "figure_captions_draft.txt"
    ),
    "",
    "IMPORTANT",
    "---------",
    (
        "All performance figures were generated from "
        "locked Route A outputs only."
    ),
    (
        "No training, checkpoint loading, raw-image access, "
        "model inference, or model selection occurred."
    ),
    (
        "Aggregation and reference sensitivity analyses are "
        "descriptive and do not alter the locked primary analysis."
    ),
]


summary_text = "\n".join(
    summary_lines
)


atomic_text_save(
    summary_text + "\n",
    OUTPUT_ROOT
    / "FIGURE_GENERATION_SUMMARY.txt",
)


print("")
print("=" * 125)
print(
    "10_route_A_manuscript_figures "
    "COMPLETED SUCCESSFULLY"
)
print("=" * 125)

print("")
print("Generated figures:")

if generated_figure_stems:
    for figure_stem in generated_figure_stems:
        print(" - " + figure_stem)

else:
    print(" - None")

print("")
print("PNG directory:")
print(PNG_DIR)

print("")
print("SVG directory:")
print(SVG_DIR)

print("")
print("PDF directory:")
print(PDF_DIR)

print("")
print("Source-data directory:")
print(SOURCE_DATA_DIR)

print("")
print("Draft captions:")
print(
    OUTPUT_ROOT
    / "figure_captions_draft.txt"
)

print("")
print("Generation manifest:")
print(
    MANIFEST_DIR
    / "figure_generation_manifest.json"
)

print("")
print("Figure-generation summary:")
print(
    OUTPUT_ROOT
    / "FIGURE_GENERATION_SUMMARY.txt"
)

print("")
print("Safety confirmation:")
print(
    "No training, checkpoint loading, raw-image access, "
    "model inference, model selection, or legacy-result use occurred."
)